# model

> Build a segmentation model (any `segmentation_models_pytorch` architecture + encoder) with configurable `in_channels` and `num_classes`, plus helpers for differential learning rates and encoder freezing.

We use `segmentation_models_pytorch`, which adapts the pretrained encoder's first convolution to any `in_channels` and lets us set `classes` to 1 (binary) or N (multi-class) directly.


In [ ]:
#| default_exp model

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import List, Optional, Set
import warnings

import torch
import torch.nn as nn


In [ ]:
#| export
import segmentation_models_pytorch as smp

## Build the model

`architecture` selects the smp segmentation head (`unet`, `fpn`, `deeplabv3plus`, …). `arch` / `encoder` selects any smp encoder backbone (`mobilenet_v2`, `resnet34`, `efficientnet-b3`, …).

`num_classes == 1` produces a single logit channel for sigmoid; `num_classes > 1` produces one channel per class for softmax. `activation=None` keeps the head returning raw logits, which is what our losses and metrics expect.


### Scratch — local ImageNet encoder weights

The Hugging Face hub cache path
`~/.cache/huggingface/hub/models--smp-hub--mobilenet_v2.imagenet` is a
**directory** (`blobs/` + `snapshots/`), not a file. `torch.load` needs a file.

Two workable paths:

1. **Online / cache already warm:** `encoder_weights='imagenet'` — SMP finds the hub snapshot itself.
2. **Portable for later (air-gapped):** save an encoder-only `.pth`, then load that file
   (same idea as `download_encoder_weights` + `get_model(weights_path=...)` below).


In [ ]:
#| eval: false
# experiment: resolve a real weight *file* (or None → let SMP use imagenet)
from pathlib import Path

HF_MOBILENET = Path.home() / '.cache/huggingface/hub/models--smp-hub--mobilenet_v2.imagenet'
# portable encoder dump you can copy to HPC later — change if you prefer another folder
PORTABLE_PTH = Path('/tmp/segmentation_tools_pt_weights/mobilenet_v2_encoder.pth')

def resolve_encoder_weight_file(hf_repo_dir: Path = HF_MOBILENET, portable: Path = PORTABLE_PTH):
    "Prefer a portable .pth; else the hub snapshot model.safetensors; else None."
    if portable.is_file():
        return portable
    if hf_repo_dir.is_dir():
        # newest snapshot's model.safetensors (HF layout)
        snaps = sorted((hf_repo_dir / 'snapshots').glob('*/model.safetensors'))
        if snaps:
            return snaps[-1]
    return None

weights_file = resolve_encoder_weight_file()
print('weights_file =', weights_file)
print('is_file =', weights_file.is_file() if weights_file else None)


In [ ]:
#| eval: false
# verify: never hand a directory to torch.load
assert weights_file is None or weights_file.is_file(), f'not a file: {weights_file}'
assert not HF_MOBILENET.is_file(), 'hub root must be a directory'
print('step ok: path is a file or None')


In [ ]:
#| eval: false
# experiment: build Unet — SMP imagenet if no portable file yet; else load encoder from file
in_channels, num_classes = 3, 1
pretrained = True

if weights_file is not None and weights_file.suffix == '.pth':
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights=None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    state = torch.load(weights_file, map_location='cpu', weights_only=False)
    missing, unexpected = model.encoder.load_state_dict(state, strict=False)
    print(f'loaded encoder .pth from {weights_file}')
    print('missing', len(missing), 'unexpected', len(unexpected))
elif weights_file is not None and weights_file.suffix == '.safetensors':
    # Hub snapshot is safetensors; easiest is still SMP's imagenet loader (uses this cache).
    # Optional: write a portable .pth for air-gap copy (next cell).
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights='imagenet' if pretrained else None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    print(f'SMP imagenet load (hub cache at {HF_MOBILENET.name})')
else:
    model = smp.Unet(
        encoder_name='mobilenet_v2', encoder_weights='imagenet' if pretrained else None,
        in_channels=in_channels, classes=num_classes, activation=None,
    )
    print('SMP imagenet download/cache')

y = model(torch.randn(1, in_channels, 64, 64))
print('out', tuple(y.shape))


In [ ]:
#| eval: false
# verify: forward shape
assert tuple(y.shape) == (1, num_classes, 64, 64)
print('forward ok')


In [ ]:
#| eval: false
# experiment: save portable encoder .pth for later / HPC (no hub directory needed)
PORTABLE_PTH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.encoder.state_dict(), PORTABLE_PTH)
print(f'saved encoder → {PORTABLE_PTH} ({PORTABLE_PTH.stat().st_size/1e6:.1f} MB)')


In [ ]:
#| eval: false
# verify: reload from portable .pth only (simulates air-gapped load)
m2 = smp.Unet(
    encoder_name='mobilenet_v2', encoder_weights=None,
    in_channels=in_channels, classes=num_classes, activation=None,
)
state2 = torch.load(PORTABLE_PTH, map_location='cpu', weights_only=False)
m2.encoder.load_state_dict(state2, strict=False)
assert tuple(m2(torch.randn(1, in_channels, 64, 64)).shape) == (1, num_classes, 64, 64)
print('portable reload ok — use this path later with get_model(weights_path=...)')
print(PORTABLE_PTH)


In [ ]:
import torch; print(torch.__version__)

In [ ]:
#| export
SMP_ARCHITECTURES = {
    'unet': smp.Unet,
    'unetplusplus': smp.UnetPlusPlus,
    'manet': smp.MAnet,
    'linknet': smp.Linknet,
    'fpn': smp.FPN,
    'pspnet': smp.PSPNet,
    'pan': smp.PAN,
    'deeplabv3': smp.DeepLabV3,
    'deeplabv3plus': smp.DeepLabV3Plus,
    'upernet': smp.UPerNet,
}


def list_smp_architectures() -> List[str]:
    """Segmentation head names accepted by `get_model(architecture=...)`."""
    return sorted(SMP_ARCHITECTURES.keys())


def list_smp_encoders() -> List[str]:
    """Encoder / backbone names from segmentation-models-pytorch (includes registered `dinov3_*`)."""
    return sorted(smp.encoders.get_encoder_names())


def _normalize_architecture(name: str) -> str:
    key = name.lower().replace('-', '').replace('_', '')
    aliases = {'unetplusplus': 'unetplusplus', 'deeplabv3plus': 'deeplabv3plus'}
    return aliases.get(key, key)


def _load_local_encoder_weights(model: nn.Module, weights_path: str, device: str) -> None:
    state = torch.load(weights_path, map_location=device, weights_only=False)
    model_state = model.encoder.state_dict()
    compatible = {
        k: v for k, v in state.items()
        if k in model_state and v.shape == model_state[k].shape
    }
    skipped = [
        k for k in state
        if k in model_state and state[k].shape != model_state[k].shape
    ]
    if skipped:
        print(
            f'WARNING: {len(skipped)} key(s) skipped from {weights_path} due to shape mismatch '
            f'(checkpoint in_channels ≠ model in_channels): {skipped}'
        )
    res = model.encoder.load_state_dict(compatible, strict=False)
    print(f'Loaded encoder from {weights_path} ({len(compatible)}/{len(model_state)} keys matched)')
    if res.missing_keys:
        print(f'  missing keys: {len(res.missing_keys)} (e.g. {res.missing_keys[0]})')
    if res.unexpected_keys:
        print(f'  unexpected keys: {len(res.unexpected_keys)} (e.g. {res.unexpected_keys[0]})')


def get_model(
    arch: str = 'mobilenet_v2',   # encoder name — alias kept for Hydra `model.arch`
    in_channels: int = 3,
    num_classes: int = 1,         # 1 = binary, >1 = multi-class
    pretrained: bool = True,      # ImageNet weights (ignored when weights_path is set)
    weights_path: Optional[str] = None,  # local encoder .pth from download_encoder_weights
    device: str = 'cpu',
    architecture: str = 'unet',   # smp head: unet | fpn | deeplabv3plus | ...
    encoder: Optional[str] = None,  # explicit encoder name; overrides `arch` when set
) -> nn.Module:
    """Build an smp model returning raw logits `(B, num_classes, H, W)`.

    Pass `weights_path` on air-gapped HPC instead of downloading ImageNet weights.
    Use `list_smp_architectures()` and `list_smp_encoders()` to discover valid names.
    `encoder='dinov3_vit_small'` / `'dinov3_vit_base'` build a DPT-reassembled DINOv3
    ViT encoder (see the DINOv3 section below) instead of a CNN backbone.

    Example:
        >>> m = get_model('resnet34', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
        >>> tuple(m(torch.randn(1, 3, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    encoder_name = encoder if encoder is not None else arch
    arch_key = _normalize_architecture(architecture)
    if arch_key not in SMP_ARCHITECTURES:
        raise ValueError(
            f"Unknown architecture {architecture!r}. "
            f"Choose one of: {', '.join(list_smp_architectures())}"
        )
    valid_encoders = smp.encoders.get_encoder_names()
    if encoder_name not in valid_encoders:
        raise ValueError(
            f"Unknown encoder {encoder_name!r}. "
            f"Use list_smp_encoders() — {len(valid_encoders)} encoders available."
        )
    use_imagenet = pretrained and weights_path is None
    model_cls = SMP_ARCHITECTURES[arch_key]
    if encoder_name.startswith('dinov3_'):
        # DINOv3 weights are downloaded inside Dinov3ReassembleEncoder.__init__ via
        # timm's own hub, not smp's encoder_weights='imagenet' state-dict loader (which
        # would `load_state_dict` the *whole* encoder, including our reassemble layers,
        # against a checkpoint that only has the plain ViT backbone -> missing keys).
        # Thread `pretrained` through the registry's construction params instead.
        smp.encoders.encoders[encoder_name]['params']['pretrained'] = use_imagenet
        encoder_weights = None
    else:
        encoder_weights = 'imagenet' if use_imagenet else None
    model = model_cls(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
    if weights_path is not None:
        _load_local_encoder_weights(model, weights_path, device)
    return model


In [ ]:
# test: output channels follow num_classes for binary and multi-class
m1 = get_model('mobilenet_v2', in_channels=2, num_classes=1, pretrained=False)
assert tuple(m1(torch.randn(1, 2, 64, 64)).shape) == (1, 1, 64, 64)
m3 = get_model('mobilenet_v2', in_channels=4, num_classes=3, pretrained=False)
assert tuple(m3(torch.randn(1, 4, 64, 64)).shape) == (1, 3, 64, 64)
m_fpn = get_model('resnet18', architecture='fpn', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_fpn(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)
assert 'resnet34' in list_smp_encoders()
assert 'deeplabv3plus' in list_smp_architectures()


In [ ]:
#| hide
#| eval: false
from pathlib import Path
weights_path = Path('/tmp/segmentation_tools_pt_weights/mobilenet_v2_encoder.pth')
print(f'Weights path exists: {weights_path.exists()} → {weights_path}')
m1 = get_model(
    'mobilenet_v2',
    in_channels=3,
    num_classes=1,
    pretrained=True,
    weights_path=str(weights_path) if weights_path.is_file() else None,
)
assert tuple(m1(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)


In [ ]:
#| eval: false
# (assert moved into the get_model scratch cell above)


## Offline / air-gapped HPC usage

On an HPC without internet, `encoder_weights='imagenet'` will raise a download error.  
The workflow is:

| Step | Where | Command |
|------|-------|---------|
| 1 | Any machine **with** internet | `download_encoder_weights('resnet34', '/tmp/resnet34.pth')` |
| 2 | Copy the `.pth` file to HPC | `scp /tmp/resnet34.pth hpc:/data/weights/` |
| 3 | HPC (no internet) | `get_model('resnet34', weights_path='/data/weights/resnet34.pth')` |

`download_encoder_weights` lets smp download + cache the weights normally, then saves **just the encoder state-dict** to a portable file.  
`get_model(weights_path=...)` builds the model with `encoder_weights=None` and loads that file — no network call is made.

In [ ]:
#| export
def download_encoder_weights(
    encoder_name: str,   # smp encoder name  (e.g. 'resnet34', 'efficientnet-b0')
    save_path: str,      # local .pth path to write   (e.g. '/data/weights/resnet34.pth')
) -> None:
    """Download ImageNet encoder weights to a portable local file.

    Run **once** on any machine that has internet access, copy the file to your
    air-gapped HPC, then use `get_model(weights_path=...)`.

    Internally builds a full smp.Unet to trigger the normal smp/timm download,
    then extracts and saves only the encoder state-dict — so the file is small
    (~20-100 MB) and independent of your `in_channels` / `num_classes` settings.

    Example:
        >>> import tempfile, os
        >>> p = os.path.join(tempfile.mkdtemp(), 'enc.pth')
        >>> download_encoder_weights('mobilenet_v2', p)   # doctest: +SKIP
        >>> os.path.exists(p)                              # doctest: +SKIP
        True
    """
    from pathlib import Path
    tmp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                   in_channels=3, classes=1, activation=None)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(tmp.encoder.state_dict(), save_path)
    n = len(tmp.encoder.state_dict())
    size_mb = Path(save_path).stat().st_size / 1e6
    print(f'Saved {encoder_name} encoder  ({n} tensors, {size_mb:.1f} MB)  →  {save_path}')
    del tmp


In [ ]:
#| eval: false
# What encoders are available in segmentation_models_pytorch (smp)?
import segmentation_models_pytorch as smp

# List all available encoders (common in stable SMP releases)
print("Available SMP encoders:")
for name in sorted(smp.encoders.get_encoder_names()):
    print("-", name)
    encoder_name = name
    try:
        download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')
    except Exception as e:
        print(f"Error downloading {encoder_name}: {e}")
    


In [ ]:
#| eval: false

# Example usage: pick one and download its weights
encoder_name = 'resnet18'  # or choose any from the printed list
download_encoder_weights(encoder_name, f'x:/Models/smp_models/{encoder_name}.pth')

## Differential learning rates and freezing

The pretrained encoder needs a gentler learning rate than the randomly-initialised decoder. Freezing the encoder for a few warm-up epochs lets the decoder catch up before fine-tuning the whole network.

In [42]:
#| export
def get_param_groups(
    model: nn.Module,        # an smp model (or any model; falls back to one group if no .encoder)
    encoder_lr: float = 1e-4,  # LR for the pretrained encoder
    decoder_lr: float = 1e-3,  # LR for the decoder + head
) -> List[dict]:
    "Two parameter groups so encoder and decoder train at different rates."
    if not hasattr(model, 'encoder'):
        return [{'params': list(model.parameters()), 'lr': decoder_lr}]
    enc = list(model.encoder.parameters())
    dec = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())
    return [{'params': enc, 'lr': encoder_lr}, {'params': dec, 'lr': decoder_lr}]

def freeze_encoder(model: nn.Module) -> None:
    "Freeze encoder weights for a decoder warm-up phase. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = False

def unfreeze_encoder(model: nn.Module) -> None:
    "Unfreeze the encoder after warm-up. No-op if model has no .encoder."
    if not hasattr(model, 'encoder'):
        return
    for p in model.encoder.parameters():
        p.requires_grad = True

## DINOv3 encoder-decoder

`list_smp_encoders()` only covers CNN-style backbones — there is no DINOv3 entry, because
DINOv3 is a plain Vision Transformer: `timm`'s `vit_*_patch16_dinov3` models emit a single
token grid at stride 16, not the stride `[2,4,8,16,32]` feature pyramid smp's `Unet`/`FPN`
decoders expect for their skip connections.

To use a DINOv3 backbone with those decoders anyway, we tap 4 evenly-spaced transformer
blocks (`timm`'s `forward_intermediates`) and reassemble each one to a different stride via
up/down-sampling convs — the same trick the DPT paper (Ranftl et al., 2021) uses to turn a
ViT into a dense-prediction backbone. ViT has no natural high-resolution feature, so the
missing stride-2 skip is synthesized by further upsampling the stride-4 map; this gives the
decoder a shallow skip for sharp boundaries instead of starting at stride 4.

The result is registered into `smp.encoders.encoders` under `dinov3_vit_small` /
`dinov3_vit_base`, so it plugs into the existing `get_model(arch=..., architecture='unet')`
call — no new API, no changes to `get_param_groups` / `freeze_encoder` / Hydra configs.

Pretrained DINOv3 weights come from `timm`'s own hub download (`pretrained=True` at
construction), not smp's `encoder_weights='imagenet'` path — that path expects to
`load_state_dict` a full encoder checkpoint with no extra reassemble layers, which would
raise missing-key errors here. `get_model()` accounts for this below.


In [37]:
#| export
import timm
from segmentation_models_pytorch.encoders._base import EncoderMixin


In [38]:
#| export

class _Reassemble(nn.Module):
    "Project a ViT token feature map to `out_ch` channels, then resize to the target stride."
    def __init__(self, in_ch: int, out_ch: int, scale: float):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        if scale > 1:
            self.resize = nn.Upsample(scale_factor=scale, mode='bilinear', align_corners=False)
        elif scale < 1:
            self.resize = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=round(1 / scale), padding=1)
        else:
            self.resize = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.resize(self.proj(x))



In [39]:
#| export
class Dinov3ReassembleEncoder(nn.Module, EncoderMixin):
    """DINOv3 ViT backbone + DPT-style reassemble, exposed as an smp `EncoderMixin`
    so it can be used as `get_model(arch='dinov3_vit_small', architecture='unet', ...)`.

    Forward returns `[x, stride2, stride4, stride8, stride16, stride32]` — 6 tensors for
    `depth=5`, matching what a CNN encoder (e.g. resnet34) would return.
    """

    def __init__(
        self,
        model_name: str = 'vit_small_patch16_dinov3',
        depth: int = 5,
        output_stride: int = 32,
        pretrained: bool = True,
        widths=(64, 96, 192, 384, 384),  # channel widths for stride 2,4,8,16,32
    ):
        super().__init__()
        self._depth = depth
        self._in_channels = 3
        self._output_stride = output_stride

        # Always built for 3-channel input; `set_in_channels()` (called by smp's
        # `get_encoder()` right after construction) adapts the patch-embed conv for
        # any other `in_channels`, reusing the same weight-averaging trick smp uses
        # for every other encoder.
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, dynamic_img_size=True,
        )
        embed_dim = self.backbone.embed_dim

        w2, w4, w8, w16, w32 = widths
        self.reassemble = nn.ModuleList([
            _Reassemble(embed_dim, w4, scale=4),    # earliest tap  -> stride 4
            _Reassemble(embed_dim, w8, scale=2),    # ..            -> stride 8
            _Reassemble(embed_dim, w16, scale=1),   # ..            -> stride 16
            _Reassemble(embed_dim, w32, scale=0.5), # last tap      -> stride 32
        ])
        # ViT has no native high-res feature -- synthesize stride 2 by upsampling
        # the stride-4 map instead of leaving the decoder's shallowest skip empty.
        self.stem_up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(w4, w2, kernel_size=3, padding=1),
        )
        self._out_channels = (3, w2, w4, w8, w16, w32)

    def set_in_channels(self, in_channels: int, pretrained: bool = True) -> None:
        "Adapt the patch-embed conv in place; `in_chans` can't be set after `timm.create_model`."
        if in_channels == 3:
            return
        from segmentation_models_pytorch.encoders._utils import patch_first_conv
        self._in_channels = in_channels
        self._out_channels = (in_channels,) + self._out_channels[1:]
        patch_first_conv(self.backbone, new_in_channels=in_channels, pretrained=pretrained)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        feats = self.backbone.forward_intermediates(
            x, indices=4, output_fmt='NCHW', intermediates_only=True,
        )
        stride4, stride8, stride16, stride32 = [r(f) for r, f in zip(self.reassemble, feats)]
        stride2 = self.stem_up(stride4)
        return [x, stride2, stride4, stride8, stride16, stride32]



In [40]:
#| export
def _register_dinov3_encoders() -> None:
    "Register the DINOv3 encoders into smp so `list_smp_encoders()` and `get_model()` see them."
    for name, model_name in [
        ('dinov3_vit_small', 'vit_small_patch16_dinov3'),
        ('dinov3_vit_base', 'vit_base_patch16_dinov3'),
    ]:
        smp.encoders.encoders[name] = {
            'encoder': Dinov3ReassembleEncoder,
            'pretrained_settings': {},  # unused -- pretrained loading happens in __init__ via timm's hub
            'params': {'model_name': model_name, 'pretrained': True},
        }

_register_dinov3_encoders()


In [43]:
# test: DINOv3 encoder is registered and pluggable through the normal get_model() call
assert {'dinov3_vit_small', 'dinov3_vit_base'} <= set(list_smp_encoders())

# small crop, no download -- pretrained=False just randomly inits the timm backbone
m_dino = get_model('dinov3_vit_small', architecture='unet', in_channels=3, num_classes=1, pretrained=False)
assert tuple(m_dino(torch.randn(1, 3, 64, 64)).shape) == (1, 1, 64, 64)

# single-channel input (grayscale X-ray, this project's common case) via set_in_channels
m_dino_gray = get_model('dinov3_vit_small', architecture='unet', in_channels=1, num_classes=1, pretrained=False)
assert tuple(m_dino_gray(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
assert m_dino_gray.encoder.backbone.patch_embed.proj.in_channels == 1

# differential LR / freeze-unfreeze already work unchanged (encoder/decoder/segmentation_head present)
groups = get_param_groups(m_dino, encoder_lr=1e-5, decoder_lr=1e-3)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m_dino)
assert all(not p.requires_grad for p in m_dino.encoder.parameters())
unfreeze_encoder(m_dino)
assert all(p.requires_grad for p in m_dino.encoder.parameters())


## Curated encoders and aliases

Problem: ConvNeXt-Tiny and EfficientNetV2-S already build via raw SMP `tu-*`
(`tu-convnext_tiny`, `tu-tf_efficientnetv2_s`), but our early `get_model` only
accepts names from `smp.encoders.get_encoder_names()`. Hydra presets and agents
cannot switch cleanly, and friendly names like `convnext_tiny` hard-error.

What we want, in order:

1. A small alias map: friendly name → curated `tu-*`.
2. Prove a curated `tu-*` forward pass works (synthetic tensor, no download).
3. Decide the policy: curated = silent; other `tu-*` = warn once; unknown non-`tu-*` = error.
4. Extend `list_smp_encoders()` so discovery shows curated aliases + raw curated `tu-*`.
5. Wire that policy into `get_model` / `download_encoder_weights`.
6. Confirm Hydra `configs/model/` packages compose (library MobileNet; multilabel DINOv3-S).


### Step 1 — alias map

Friendly Hydra names should resolve to the SMP `tu-*` strings we already know work.


In [44]:
# experiment: alias map (not exported yet)
aliases = {
    'convnext_tiny': 'tu-convnext_tiny',
    'efficientnetv2_s': 'tu-tf_efficientnetv2_s',
}
for friendly, raw in aliases.items():
    print(f'{friendly} -> {raw}')


convnext_tiny -> tu-convnext_tiny
efficientnetv2_s -> tu-tf_efficientnetv2_s


In [45]:
# verify: every friendly name maps to a tu-* string
assert aliases['convnext_tiny'].startswith('tu-')
assert aliases['efficientnetv2_s'].startswith('tu-')
assert set(aliases) == {'convnext_tiny', 'efficientnetv2_s'}
print('step 1 ok')


step 1 ok


### Step 2 — curated `tu-*` builds

SMP accepts these even when they are missing from `get_encoder_names()`. Check a
tiny forward pass with `pretrained=False` (no download).


In [46]:
# experiment: raw curated tu-* through smp.Unet
x = torch.randn(1, 1, 64, 64)
for enc in aliases.values():
    m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=1, classes=1, activation=None)
    y = m(x)
    print(enc, tuple(y.shape))


tu-convnext_tiny (1, 1, 64, 64)
tu-tf_efficientnetv2_s (1, 1, 64, 64)


In [47]:
# verify: output spatial size matches input; one logit channel
x = torch.randn(1, 1, 64, 64)
for enc in aliases.values():
    m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=1, classes=1, activation=None)
    assert tuple(m(x).shape) == (1, 1, 64, 64), enc
print('step 2 ok')


step 2 ok


### Step 3 — allowlist policy

- Name in SMP registry or curated `tu-*` → accept.
- Other `tu-*` → `UserWarning` once, then accept (experimental).
- Anything else → `ValueError`.


In [48]:
# experiment: policy sketched inline (not exported yet)
import warnings

curated_tu = set(aliases.values())
_warned = set()

def check_encoder(name: str) -> None:
    registry = set(smp.encoders.get_encoder_names())
    if name in registry or name in curated_tu:
        return
    if name.startswith('tu-'):
        if name not in _warned:
            warnings.warn(f'experimental tu-* encoder: {name}', UserWarning)
            _warned.add(name)
        return
    raise ValueError(f'Unknown encoder {name!r}')

print('check_encoder ready; curated_tu =', sorted(curated_tu))


check_encoder ready; curated_tu = ['tu-convnext_tiny', 'tu-tf_efficientnetv2_s']


In [49]:
# verify: allowlist policy (curated silent; experimental warn-once; unknown errors)
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    check_encoder('tu-convnext_tiny')
    assert not w, w

with warnings.catch_warnings(record=True) as w1:
    warnings.simplefilter('always')
    check_encoder('tu-mobilenetv3_small_100')
    assert any('experimental' in str(x.message) for x in w1), w1
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter('always')
    check_encoder('tu-mobilenetv3_small_100')
    assert not [x for x in w2 if 'experimental' in str(x.message)], w2

try:
    check_encoder('not_a_real_encoder_xyz')
    assert False, 'expected ValueError'
except ValueError as e:
    assert 'not_a_real_encoder_xyz' in str(e)
print('step 3 ok')


step 3 ok


### Step 4 — discovery list

`list_smp_encoders()` should advertise curated aliases and raw curated `tu-*`
alongside the SMP registry (including `dinov3_*` after registration above).
Experimental `tu-*` stay buildable but are not listed.


In [50]:
# experiment: discovery = registry ∪ aliases ∪ curated tu-*
discovered = set(smp.encoders.get_encoder_names())
discovered.update(aliases.keys())
discovered.update(aliases.values())
print('convnext_tiny' in discovered, 'tu-convnext_tiny' in discovered)
print('dinov3_vit_small' in discovered, 'mobilenet_v2' in discovered)
print('tu-mobilenetv3_small_100' in discovered)  # experimental: should stay False


True True
True True
False


In [51]:
# verify: curated advertised; experimental not listed
discovered = set(smp.encoders.get_encoder_names()) | set(aliases) | set(aliases.values())
assert {'convnext_tiny', 'efficientnetv2_s', 'tu-convnext_tiny', 'tu-tf_efficientnetv2_s'} <= discovered
assert {'mobilenet_v2', 'dinov3_vit_small'} <= discovered
assert 'tu-mobilenetv3_small_100' not in discovered
print('step 4 ok')


step 4 ok


### Promote — export the working pieces

Steps 1–4 passed on synthetic checks. Wrap them as library functions and thread
the policy through `list_smp_encoders`, `get_model`, and `download_encoder_weights`
(nbdev: later `#| export` of the same name replaces the earlier one).


In [52]:
#| export
CURATED_ENCODER_ALIASES = {
    'convnext_tiny': 'tu-convnext_tiny',
    'efficientnetv2_s': 'tu-tf_efficientnetv2_s',
}
CURATED_TU_ENCODERS = set(CURATED_ENCODER_ALIASES.values())
_warned_experimental_tu: Set[str] = set()


def resolve_encoder_name(name: str) -> str:
    "Map a curated alias to its SMP `tu-*` name; pass other names through."
    return CURATED_ENCODER_ALIASES.get(name, name)


def _validate_encoder_name(encoder_name: str) -> None:
    "Accept SMP registry / curated `tu-*`; warn once for experimental `tu-*`; else error."
    registry = set(smp.encoders.get_encoder_names())
    if encoder_name in registry or encoder_name in CURATED_TU_ENCODERS:
        return
    if encoder_name.startswith('tu-'):
        if encoder_name not in _warned_experimental_tu:
            warnings.warn(
                f'Encoder {encoder_name!r} is an experimental tu-* (timm) encoder — '
                f'accepted, but not curated. Prefer {sorted(CURATED_ENCODER_ALIASES)} '
                f'or raw curated names {sorted(CURATED_TU_ENCODERS)}.',
                UserWarning,
                stacklevel=3,
            )
            _warned_experimental_tu.add(encoder_name)
        return
    raise ValueError(
        f'Unknown encoder {encoder_name!r}. '
        f'Use list_smp_encoders() for curated + SMP names, or a tu-* timm encoder.'
    )


In [53]:
#| export

def list_smp_encoders() -> List[str]:
    """Encoder names for discovery: SMP registry + DINOv3 + curated aliases / `tu-*`.

    Experimental (non-curated) `tu-*` encoders are accepted by `get_model` with a
    warning but are **not** listed here — use a curated name or a raw `tu-*` knowingly.
    """
    names = set(smp.encoders.get_encoder_names())
    names.update(CURATED_ENCODER_ALIASES.keys())
    names.update(CURATED_TU_ENCODERS)
    return sorted(names)



In [54]:
list_smp_encoders()

['convnext_tiny',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'dinov3_vit_base',
 'dinov3_vit_small',
 'dpn107',
 'dpn131',
 'dpn68',
 'dpn68b',
 'dpn92',
 'dpn98',
 'efficientnet-b0',
 'efficientnet-b1',
 'efficientnet-b2',
 'efficientnet-b3',
 'efficientnet-b4',
 'efficientnet-b5',
 'efficientnet-b6',
 'efficientnet-b7',
 'efficientnetv2_s',
 'inceptionresnetv2',
 'inceptionv4',
 'mit_b0',
 'mit_b1',
 'mit_b2',
 'mit_b3',
 'mit_b4',
 'mit_b5',
 'mobilenet_v2',
 'mobileone_s0',
 'mobileone_s1',
 'mobileone_s2',
 'mobileone_s3',
 'mobileone_s4',
 'resnet101',
 'resnet152',
 'resnet18',
 'resnet34',
 'resnet50',
 'resnext101_32x16d',
 'resnext101_32x32d',
 'resnext101_32x48d',
 'resnext101_32x4d',
 'resnext101_32x8d',
 'resnext50_32x4d',
 'se_resnet101',
 'se_resnet152',
 'se_resnet50',
 'se_resnext101_32x4d',
 'se_resnext50_32x4d',
 'senet154',
 'timm-efficientnet-b0',
 'timm-efficientnet-b1',
 'timm-efficientnet-b2',
 'timm-efficientnet-b3',
 'timm-efficientnet-b4

In [57]:
#| export
def get_model(
    arch: str = 'mobilenet_v2',   # encoder name — alias kept for Hydra `model.arch`
    in_channels: int = 3,
    num_classes: int = 1,         # 1 = binary, >1 = multi-class
    pretrained: bool = True,      # ImageNet weights (ignored when weights_path is set)
    weights_path: Optional[str] = None,  # local encoder .pth from download_encoder_weights
    device: str = 'cpu',
    architecture: str = 'unet',   # smp head: unet | fpn | deeplabv3plus | ...
    encoder: Optional[str] = None,  # explicit encoder name; overrides `arch` when set
) -> nn.Module:
    """Build an smp model returning raw logits `(B, num_classes, H, W)`.

    Pass `weights_path` on air-gapped HPC instead of downloading ImageNet weights.
    Use `list_smp_architectures()` and `list_smp_encoders()` to discover valid names.
    Curated aliases: `convnext_tiny` → `tu-convnext_tiny`, `efficientnetv2_s` →
    `tu-tf_efficientnetv2_s`. Other `tu-*` names warn once then build. Unknown
    non-`tu-*` names raise. `dinov3_vit_small` / `dinov3_vit_base` use the DPT
    reassemble path below.

    Example:
        >>> m = get_model('convnext_tiny', in_channels=1, num_classes=1, pretrained=False)
        >>> tuple(m(torch.randn(1, 1, 64, 64)).shape)
        (1, 1, 64, 64)
    """
    encoder_name = resolve_encoder_name(encoder if encoder is not None else arch)
    arch_key = _normalize_architecture(architecture)
    if arch_key not in SMP_ARCHITECTURES:
        raise ValueError(
            f"Unknown architecture {architecture!r}. "
            f"Choose one of: {', '.join(list_smp_architectures())}"
        )
    _validate_encoder_name(encoder_name)
    use_imagenet = pretrained and weights_path is None
    model_cls = SMP_ARCHITECTURES[arch_key]
    if encoder_name.startswith('dinov3_'):
        smp.encoders.encoders[encoder_name]['params']['pretrained'] = use_imagenet
        encoder_weights = None
    else:
        encoder_weights = 'imagenet' if use_imagenet else None
    model = model_cls(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=num_classes,
        activation=None,
    )
    if weights_path is not None:
        _load_local_encoder_weights(model, weights_path, device)
    return model



In [58]:
mdl = get_model('dinov3_vit_small', in_channels=1, num_classes=1, pretrained=True)

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.4MB            

model.safetensors: downloading bytes:           |  0.00B            

In [60]:
#| export
def download_encoder_weights(
    encoder_name: str,   # smp encoder name or curated alias
    save_path: str,      # local .pth path to write
) -> None:
    """Download ImageNet encoder weights to a portable local file.

    Run **once** on any machine that has internet access, copy the file to your
    air-gapped HPC, then use `get_model(weights_path=...)`. Curated aliases
    (`convnext_tiny`, …) are resolved to their `tu-*` names first.

    Example:
        >>> import tempfile, os
        >>> p = os.path.join(tempfile.mkdtemp(), 'enc.pth')
        >>> download_encoder_weights('mobilenet_v2', p)   # doctest: +SKIP
        >>> os.path.exists(p)                              # doctest: +SKIP
        True
    """
    from pathlib import Path
    encoder_name = resolve_encoder_name(encoder_name)
    _validate_encoder_name(encoder_name)
    tmp = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                   in_channels=3, classes=1, activation=None)
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(tmp.encoder.state_dict(), save_path)
    n = len(tmp.encoder.state_dict())
    size_mb = Path(save_path).stat().st_size / 1e6
    print(f'Saved {encoder_name} encoder  ({n} tensors, {size_mb:.1f} MB)  →  {save_path}')
    del tmp


In [61]:
# test: curated get_model / list / policy (same checks as steps 1–4, via exports)
import warnings

assert resolve_encoder_name('convnext_tiny') == 'tu-convnext_tiny'
assert resolve_encoder_name('mobilenet_v2') == 'mobilenet_v2'

for name in ('convnext_tiny', 'tu-convnext_tiny', 'efficientnetv2_s', 'tu-tf_efficientnetv2_s'):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        m = get_model(name, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
        assert tuple(m(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64), name
        tu_warns = [x for x in w if issubclass(x.category, UserWarning) and 'experimental' in str(x.message).lower()]
        assert not tu_warns, f'unexpected warning for curated {name}: {tu_warns}'

encs = set(list_smp_encoders())
assert {'convnext_tiny', 'efficientnetv2_s', 'tu-convnext_tiny', 'tu-tf_efficientnetv2_s'} <= encs
assert {'mobilenet_v2', 'dinov3_vit_small'} <= encs

experimental = 'tu-mobilenetv3_small_100'
with warnings.catch_warnings(record=True) as w1:
    warnings.simplefilter('always')
    m_exp = get_model(experimental, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
    assert tuple(m_exp(torch.randn(1, 1, 64, 64)).shape) == (1, 1, 64, 64)
    assert any('experimental' in str(x.message).lower() and experimental in str(x.message) for x in w1)
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter('always')
    get_model(experimental, architecture='unet', in_channels=1, num_classes=1, pretrained=False)
    assert not [x for x in w2 if experimental in str(x.message)]

try:
    get_model('not_a_real_encoder_xyz', pretrained=False)
    assert False, 'expected ValueError'
except ValueError as e:
    assert 'not_a_real_encoder_xyz' in str(e)


### Hydra packages and ONNX smoke

Config packages live under `configs/model/` (YAML only). Compose checks belong
next to the factory; ONNX smoke uses the curated `get_model` path for CNN aliases
(`dinov3_*` ONNX is heavier — covered elsewhere).


In [62]:
# experiment: Hydra model packages compose (not exported — YAML only)
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
assert cfg_dir is not None, f'configs/ not found from cwd={here}'
GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    lib = compose(config_name='train')
    ml = compose(config_name='train_multilabel_3head')
    print('library default arch:', lib.model.arch)
    print('multilabel recipe arch:', ml.model.arch)
    print('shut-off from multilabel:', compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2']).model.arch)


library default arch: mobilenet_v2
multilabel recipe arch: dinov3_vit_small
shut-off from multilabel: mobilenet_v2


In [63]:
# verify: library MobileNet; multilabel DINOv3-S; shut-off MobileNet; aliases compose
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    assert compose(config_name='train').model.arch == 'mobilenet_v2'
    assert compose(config_name='train_multilabel_3head').model.arch == 'dinov3_vit_small'
    assert compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2']).model.arch == 'mobilenet_v2'
    assert compose(config_name='train', overrides=['model=convnext_tiny']).model.arch == 'convnext_tiny'
    assert compose(config_name='train', overrides=['model=efficientnetv2_s']).model.arch == 'efficientnetv2_s'
print('hydra compose ok')


hydra compose ok


In [64]:
# test: HYDRA_ENCODER_PRESET_COMPOSE
from pathlib import Path
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

here = Path.cwd()
cfg_dir = next(
    (str(p.resolve()) for p in (here / 'configs', here.parent / 'configs') if p.is_dir()),
    None,
)
assert cfg_dir is not None, f'configs/ not found from cwd={here}'

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=cfg_dir, version_base='1.3'):
    cfg = compose(config_name='train')
    assert cfg.model.arch == 'mobilenet_v2'
    assert cfg.model.architecture == 'unet'
    cfg_dino = compose(config_name='train', overrides=['model=dinov3_vit_small'])
    assert cfg_dino.model.arch == 'dinov3_vit_small'
    cfg_cn = compose(config_name='train', overrides=['model=convnext_tiny'])
    assert cfg_cn.model.arch == 'convnext_tiny'
    cfg_eff = compose(config_name='train', overrides=['model=efficientnetv2_s'])
    assert cfg_eff.model.arch == 'efficientnetv2_s'
    cfg_ml = compose(config_name='train_multilabel_3head')
    assert cfg_ml.model.arch == 'dinov3_vit_small'
    cfg_off = compose(config_name='train_multilabel_3head', overrides=['model=mobilenet_v2'])
    assert cfg_off.model.arch == 'mobilenet_v2'


In [65]:
# test: CURATED_ENCODER_ONNX_SMOKE
# FP32 ONNX smoke for curated CNN builds via get_model (fixed spatial size).
# dinov3_* ONNX is covered separately (pretrained=False still heavy); skip here.
import tempfile
from pathlib import Path
from segmentation_tools_pt.export import export_onnx

for name in ('mobilenet_v2', 'convnext_tiny', 'efficientnetv2_s'):
    m = get_model(name, architecture='unet', in_channels=1, num_classes=3, pretrained=False)
    with tempfile.TemporaryDirectory() as d:
        out = Path(d) / f'{name}.onnx'
        export_onnx(m, str(out), in_channels=1, input_size=64, opset=18, verify=True)
        assert out.exists() and out.stat().st_size > 0, name


In [66]:
# test
groups = get_param_groups(m1)
assert len(groups) == 2 and groups[0]['lr'] < groups[1]['lr']
freeze_encoder(m1)
assert all(not p.requires_grad for p in m1.encoder.parameters())
unfreeze_encoder(m1)
assert all(p.requires_grad for p in m1.encoder.parameters())

# fallback: a plain (non-SMP) model has no .encoder -- one param group, freeze/unfreeze are no-ops
toy = nn.Conv2d(3, 1, 3, padding=1)
toy_groups = get_param_groups(toy)
assert len(toy_groups) == 1 and toy_groups[0]['lr'] == 1e-3
assert set(toy_groups[0]['params']) == set(toy.parameters())
freeze_encoder(toy)   # no .encoder -> no-op, must not raise
unfreeze_encoder(toy)
assert all(p.requires_grad for p in toy.parameters())

In [68]:
#| hide
import nbdev; nbdev.nbdev_export()